# 06. 파손 데이터 비교 — 05번 top2 교차 실험 (6번 / 7번 공용)

두 데이터셋을 **같은 노트북**으로 돌립니다.
맨 위 설정 셀의 `DATA_IS_ONLY_CLEAN` 하나만 바꾸면 됩니다.

| 스위치 | 데이터셋 | 출력 폴더 |
|---|---|---|
| `DATA_IS_ONLY_CLEAN = True` | clean 데이터만 (약 7,600장) | `06_1_only_clean_experiment` |
| `DATA_IS_ONLY_CLEAN = False` | clean 중 20%를 damage로 교체 (약 7,600장) | `06_2_clean+20percent_damaged_mixed_experiment` |

두 데이터셋의 **장수가 같으므로**, 차이가 나면 그건 데이터 양이 아니라
**파손 이미지가 섞인 효과**입니다. 이게 이 비교의 핵심입니다.

`data/processed` 경로는 그대로이므로, 실행 전에 해당 데이터로 덮어쓰기만 하면 됩니다.

## 8가지 실험 (모두 15 epoch)

05번이 남긴 **하이퍼파라미터 top2**와 **증강 top2**를 교차합니다.

### baseline 4가지

| ID | 증강 | 하이퍼파라미터 |
|----|------|----------------|
| B00 | 강제 전부 0 | auto |
| B01 | YOLO 기본 | auto |
| B02 | YOLO 기본 | **hp1** |
| B03 | 강제 전부 0 | **hp1** |

`B00`/`B01`/`B02`/`B03`이 증강 x 하이퍼파라미터 2x2 대조를 이룹니다.

### 교차 4가지

| 증강 | hp1 | hp2 |
|------|-----|-----|
| **aug1** | `aug1 x hp1` | `aug1 x hp2` |
| **aug2** | `aug2 x hp1` | `aug2 x hp2` |

이렇게 하면 "증강을 바꾼 효과"와 "하이퍼파라미터를 바꾼 효과"를
한 표 안에서 각각 읽을 수 있습니다.

## 주의: 다른 노트북 숫자와 직접 비교하지 마세요

학습 데이터가 다릅니다. 같은 ID끼리(B00 vs B00) 볼 수는 있지만
데이터가 다르다는 점을 반드시 밝히세요.


## 1. 라이브러리

In [ ]:
from __future__ import annotations

import json
import math
import time
import random
import hashlib
import shutil
import gc
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import yaml
import torch

from ultralytics import YOLO

try:
    import ultralytics
    ULTRALYTICS_VERSION = ultralytics.__version__
except Exception:
    ULTRALYTICS_VERSION = "unknown"

try:
    from ultralytics.cfg import DEFAULT_CFG_DICT
except Exception:
    DEFAULT_CFG_DICT = {}

from IPython.display import display, Image as IPImage

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 500)


def set_korean_font():
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
    installed = {font.name for font in fm.fontManager.ttflist}

    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break

    plt.rcParams["axes.unicode_minus"] = False


set_korean_font()

print("Ultralytics:", ULTRALYTICS_VERSION)
print("PyTorch    :", torch.__version__)
print("OpenCV     :", cv2.__version__)

## 2. 경로와 공통 설정

**여기서 `DATA_IS_ONLY_CLEAN`을 바꿉니다.**

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# ------------------------------------------------------------
# 1) 1번 노트북이 만든 processed 데이터
# ------------------------------------------------------------
PROCESSED_DIR = (
    PROJECT_ROOT / "../../data/processed"
).resolve()

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"processed 폴더가 없습니다: {PROCESSED_DIR}\n"
        "먼저 01_recycling_eda_preprocess_build_processed.ipynb를 실행하세요."
    )

DATA_YAML = PROCESSED_DIR / "data.yaml"

# ------------------------------------------------------------
# 2) 모델/실험 산출물 위치
# ------------------------------------------------------------
# ------------------------------------------------------------
# 어떤 데이터셋인지 고르는 스위치
# ------------------------------------------------------------
#   True  : clean 데이터만 (약 7,600장)
#   False : clean 중 20%를 damage로 교체 (데이터 수 동일, 약 7,600장)
DATA_IS_ONLY_CLEAN = True

if DATA_IS_ONLY_CLEAN:
    EXPERIMENT_FOLDER = "06_1_only_clean_experiment"
    DATASET_LABEL = "only clean"
else:
    EXPERIMENT_FOLDER = "06_2_clean+20percent_damaged_mixed_experiment"
    DATASET_LABEL = "clean + 20% damage 교체"

EXPERIMENT_ROOT = (
    PROJECT_ROOT / f"../models/yolo/{EXPERIMENT_FOLDER}"
).resolve()

RUNS_DIR = EXPERIMENT_ROOT / "runs"
CV_CACHE_DIR = EXPERIMENT_ROOT / "opencv_datasets"

# ------------------------------------------------------------
# 3) 사람이 확인할 보고서 위치
# ------------------------------------------------------------
REPORT_ROOT = EXPERIMENT_ROOT / "report"
REPORT_SOURCE_DIR = REPORT_ROOT / "preprocess"
SUMMARY_DIR = REPORT_ROOT / "summary"
PER_CLASS_DIR = REPORT_ROOT / "per_class"
FINAL_DIR = REPORT_ROOT / "final_best"

for path in [
    EXPERIMENT_ROOT,
    RUNS_DIR,
    CV_CACHE_DIR,
    REPORT_ROOT,
    REPORT_SOURCE_DIR,
    SUMMARY_DIR,
    PER_CLASS_DIR,
    FINAL_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "yolo26n.pt"
EPOCHS = 15
IMGSZ = 640
BATCH = 8
PATIENCE = 12

# close_mosaic 은 '마지막 N epoch 동안 mosaic/mixup/cutmix 를 끈다'는 뜻입니다.
# Ultralytics 기본값 10을 15 epoch 학습에 쓰면 앞 5 epoch 만 mosaic 이 적용됩니다.
# 05번과 같은 3 으로 맞춰, 마지막 구간에서만 끄도록 합니다.
CLOSE_MOSAIC_EPOCHS = 3
# 데이터 로딩을 별도 프로세스로 병렬화합니다 (물리 14코어).
# 46개 런 전부 같은 값으로 돌려야 실험 간 비교가 공정합니다.
WORKERS = 4
SEED = 42

RUN_EXPERIMENTS = True
RUN_MULTI_SEED = False
SKIP_COMPLETED = True
SMOKE_TEST = False

# OpenCV static augmentation은 원본과 같은 데이터 개수를 유지합니다.
CV_APPLY_PROBABILITY = 0.80
CV_OUTPUT_JPEG_QUALITY = 95
CLEANUP_CV_DATASET_AFTER_RUN = True

if SMOKE_TEST:
    EPOCHS = 2
    PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("PROCESSED_DIR  :", PROCESSED_DIR)
print("데이터셋       :", DATASET_LABEL)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("RUNS_DIR       :", RUNS_DIR)
print("REPORT_ROOT    :", REPORT_ROOT)
print("SUMMARY_DIR    :", SUMMARY_DIR)
print("MODEL_NAME     :", MODEL_NAME)
print("EPOCHS         :", EPOCHS)
print("IMGSZ          :", IMGSZ)
print("BATCH          :", BATCH)
print("SMOKE_TEST     :", SMOKE_TEST)


## 3. GPU 확인

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {total_vram:.2f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS 사용")
else:
    DEVICE = "cpu"
    print("CPU 사용")

print("DEVICE =", DEVICE)

## 4. data.yaml 보정

클래스 수가 17인지 출력으로 확인하세요.

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

dataset_config["path"] = str(PROCESSED_DIR.resolve())

RUNTIME_DATA_YAML = SUMMARY_DIR / "runtime_processed.yaml"

with open(RUNTIME_DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, allow_unicode=True, sort_keys=False)

names_raw = dataset_config["names"]

if isinstance(names_raw, dict):
    CLASS_NAMES = {int(key): str(value) for key, value in names_raw.items()}
else:
    CLASS_NAMES = {index: str(value) for index, value in enumerate(names_raw)}

NUM_CLASSES = len(CLASS_NAMES)

print(RUNTIME_DATA_YAML.read_text(encoding="utf-8")[:5000])
print("클래스 수:", NUM_CLASSES)

## 5. 품질 보고서 (있으면)

In [ ]:
CLASS_SUPPORT_CSV = REPORT_SOURCE_DIR / "class_support_processed.csv"
PREPROCESS_SUMMARY_CSV = REPORT_SOURCE_DIR / "preprocess_summary.csv"

if CLASS_SUPPORT_CSV.exists():
    class_support_df = pd.read_csv(CLASS_SUPPORT_CSV)
    display(class_support_df)

    no_val_classes = class_support_df[class_support_df["val"].eq(0)]
    print("Validation object가 0인 클래스:", len(no_val_classes))
    if len(no_val_classes):
        display(no_val_classes[["class_name", "train", "val"]])
else:
    class_support_df = pd.DataFrame()
    print("class_support_processed.csv를 찾지 못했습니다.")

if PREPROCESS_SUMMARY_CSV.exists():
    display(pd.read_csv(PREPROCESS_SUMMARY_CSV))

## 6. 데이터 확인

**두 데이터셋의 train 장수가 비슷해야 합니다(약 7,600장).**
한쪽이 15,000장대로 나오면 2배 데이터가 들어와 있는 것이니 교체하세요.

In [ ]:
TRAIN_IMAGE_DIR = PROCESSED_DIR / "images" / "train"
VAL_IMAGE_DIR = PROCESSED_DIR / "images" / "val"
TRAIN_LABEL_DIR = PROCESSED_DIR / "labels" / "train"
VAL_LABEL_DIR = PROCESSED_DIR / "labels" / "val"

for path in [TRAIN_IMAGE_DIR, VAL_IMAGE_DIR, TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)

train_images = sorted(path for path in TRAIN_IMAGE_DIR.iterdir() if path.is_file())
val_images = sorted(path for path in VAL_IMAGE_DIR.iterdir() if path.is_file())
train_labels = sorted(TRAIN_LABEL_DIR.glob("*.txt"))
val_labels = sorted(VAL_LABEL_DIR.glob("*.txt"))

print("Train images:", len(train_images))
print("Train labels:", len(train_labels))
print("Val images  :", len(val_images))
print("Val labels  :", len(val_labels))

if len(train_images) != len(train_labels):
    raise ValueError("Train image와 label 개수가 다릅니다.")

if len(val_images) != len(val_labels):
    raise ValueError("Validation image와 label 개수가 다릅니다.")

if len(val_images) == 0:
    raise ValueError("Validation 이미지가 없습니다.")

print("Processed quick check: PASSED")

## 7. 증강 설정 정의

In [ ]:
AUGMENTATION_KEYS = [
    "hsv_h", "hsv_s", "hsv_v",
    "degrees", "translate", "scale", "shear", "perspective",
    "flipud", "fliplr", "bgr",
    "mosaic", "mixup", "cutmix", "copy_paste",
    "close_mosaic", "augmentations",
]

installed_aug_defaults = {
    key: DEFAULT_CFG_DICT.get(key, "<not available>")
    for key in AUGMENTATION_KEYS
}

display(
    pd.DataFrame({
        "argument": list(installed_aug_defaults.keys()),
        "installed_default": list(installed_aug_defaults.values()),
    })
)

In [ ]:
NO_AUG = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "bgr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 0,
    "augmentations": [],
}


def with_no_aug(**changes):
    config = dict(NO_AUG)
    config.update(changes)
    return config


def supported_train_args(config: dict | None):
    """현재 Ultralytics 버전에서 지원되는 key만 남깁니다."""
    if config is None:
        return {}, []

    if not DEFAULT_CFG_DICT:
        # config dictionary를 가져오지 못한 버전에서는 그대로 전달합니다.
        return dict(config), []

    supported = set(DEFAULT_CFG_DICT.keys())
    unknown = sorted(set(config.keys()) - supported)
    filtered = {key: value for key, value in config.items() if key in supported}

    return filtered, unknown

In [ ]:
YOLO_AUG_CONFIGS = {
    "hsv": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.50,
        hsv_v=0.35,
    ),
    "flip": with_no_aug(
        fliplr=0.50,
    ),
    "rotation": with_no_aug(
        degrees=10.0,
    ),
    "translate": with_no_aug(
        translate=0.08,
    ),
    "scale": with_no_aug(
        scale=0.25,
    ),
    "shear": with_no_aug(
        shear=2.0,
    ),
    "perspective": with_no_aug(
        perspective=0.0005,
    ),
    "mosaic": with_no_aug(
        mosaic=0.70,
        close_mosaic=CLOSE_MOSAIC_EPOCHS,
    ),
    "mixup": with_no_aug(
        mixup=0.15,
        close_mosaic=CLOSE_MOSAIC_EPOCHS,
    ),
    "cutmix": with_no_aug(
        cutmix=0.15,
        close_mosaic=CLOSE_MOSAIC_EPOCHS,
    ),
    "hsv_flip": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.50,
        hsv_v=0.35,
        fliplr=0.50,
    ),
    "geo_combo": with_no_aug(
        degrees=10.0,
        translate=0.08,
        scale=0.25,
        shear=2.0,
        perspective=0.0005,
        fliplr=0.50,
    ),
    "mosaic_hsv_geo": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.45,
        hsv_v=0.30,
        degrees=8.0,
        translate=0.06,
        scale=0.20,
        fliplr=0.50,
        mosaic=0.65,
        close_mosaic=CLOSE_MOSAIC_EPOCHS,
    ),
    "mosaic_mixup_cutmix": with_no_aug(
        mosaic=0.65,
        mixup=0.10,
        cutmix=0.10,
        close_mosaic=CLOSE_MOSAIC_EPOCHS,
    ),
    "balanced_combo": with_no_aug(
        hsv_h=0.012,
        hsv_s=0.40,
        hsv_v=0.30,
        degrees=7.0,
        translate=0.06,
        scale=0.18,
        shear=1.0,
        perspective=0.0003,
        fliplr=0.45,
        mosaic=0.50,
        mixup=0.05,
        cutmix=0.05,
        close_mosaic=CLOSE_MOSAIC_EPOCHS,
    ),
}

## 8. OpenCV 증강 함수와 static dataset 생성기

05번이 고른 증강이 OpenCV 계열(C/H)일 수 있으므로 그대로 가져옵니다.

In [ ]:
def image_to_label_path(image_path: Path) -> Path:
    parts = list(image_path.parts)
    indices = [index for index, part in enumerate(parts) if part.lower() == "images"]

    if not indices:
        raise ValueError(f"images 폴더가 경로에 없습니다: {image_path}")

    parts[indices[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


def read_yolo_label(label_path: Path, image_width: int, image_height: int):
    classes = []
    boxes = []

    text = label_path.read_text(encoding="utf-8").strip()

    for line in text.splitlines():
        class_id, xc, yc, bw, bh = map(float, line.split())
        class_id = int(class_id)

        x1 = (xc - bw / 2) * image_width
        y1 = (yc - bh / 2) * image_height
        x2 = (xc + bw / 2) * image_width
        y2 = (yc + bh / 2) * image_height

        classes.append(class_id)
        boxes.append([x1, y1, x2, y2])

    return (
        np.asarray(classes, dtype=int),
        np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
    )


def boxes_to_yolo_lines(classes, boxes, image_width: int, image_height: int):
    lines = []

    for class_id, box in zip(classes, boxes):
        x1, y1, x2, y2 = map(float, box)

        xc = ((x1 + x2) / 2) / image_width
        yc = ((y1 + y2) / 2) / image_height
        bw = (x2 - x1) / image_width
        bh = (y2 - y1) / image_height

        lines.append(
            f"{int(class_id)} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}"
        )

    return lines

In [ ]:
def bbox_area(boxes: np.ndarray):
    if len(boxes) == 0:
        return np.empty((0,), dtype=np.float32)

    widths = np.clip(boxes[:, 2] - boxes[:, 0], 0, None)
    heights = np.clip(boxes[:, 3] - boxes[:, 1], 0, None)
    return widths * heights


def transform_boxes(
    boxes: np.ndarray,
    matrix: np.ndarray,
    image_width: int,
    image_height: int,
    min_visible: float = 0.25,
    min_size: float = 2.0,
):
    if len(boxes) == 0:
        return boxes.copy(), np.empty((0,), dtype=bool)

    corners = np.stack(
        [
            boxes[:, [0, 1]],
            boxes[:, [2, 1]],
            boxes[:, [2, 3]],
            boxes[:, [0, 3]],
        ],
        axis=1,
    ).astype(np.float32)

    points = corners.reshape(-1, 1, 2)

    if matrix.shape == (2, 3):
        transformed = cv2.transform(points, matrix).reshape(-1, 4, 2)
    else:
        transformed = cv2.perspectiveTransform(points, matrix).reshape(-1, 4, 2)

    raw_boxes = np.column_stack([
        transformed[:, :, 0].min(axis=1),
        transformed[:, :, 1].min(axis=1),
        transformed[:, :, 0].max(axis=1),
        transformed[:, :, 1].max(axis=1),
    ]).astype(np.float32)

    raw_area = bbox_area(raw_boxes)

    clipped = raw_boxes.copy()
    clipped[:, [0, 2]] = np.clip(clipped[:, [0, 2]], 0, image_width)
    clipped[:, [1, 3]] = np.clip(clipped[:, [1, 3]], 0, image_height)

    clipped_area = bbox_area(clipped)
    visible_ratio = clipped_area / np.maximum(raw_area, 1e-6)

    keep = (
        ((clipped[:, 2] - clipped[:, 0]) >= min_size)
        & ((clipped[:, 3] - clipped[:, 1]) >= min_size)
        & (visible_ratio >= min_visible)
    )

    return clipped[keep], keep

In [ ]:
def cv_brightness_contrast(image, boxes, classes, rng):
    alpha = float(rng.uniform(0.75, 1.25))
    beta = float(rng.uniform(-30, 30))

    output = np.clip(
        image.astype(np.float32) * alpha + beta,
        0,
        255,
    ).astype(np.uint8)

    return output, boxes.copy(), classes.copy()


def cv_gamma(image, boxes, classes, rng):
    gamma = float(rng.uniform(0.70, 1.40))
    inverse_gamma = 1.0 / gamma

    table = np.array([
        ((value / 255.0) ** inverse_gamma) * 255
        for value in np.arange(256)
    ]).astype(np.uint8)

    output = cv2.LUT(image, table)
    return output, boxes.copy(), classes.copy()


def cv_clahe(image, boxes, classes, rng):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=float(rng.uniform(1.5, 3.0)),
        tileGridSize=(8, 8),
    )

    enhanced_l = clahe.apply(l_channel)
    output = cv2.cvtColor(
        cv2.merge([enhanced_l, a_channel, b_channel]),
        cv2.COLOR_LAB2BGR,
    )

    return output, boxes.copy(), classes.copy()


def cv_gaussian_blur(image, boxes, classes, rng):
    kernel_size = int(rng.choice([3, 5]))
    output = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return output, boxes.copy(), classes.copy()


def cv_motion_blur(image, boxes, classes, rng):
    kernel_size = int(rng.choice([5, 7]))
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    direction = int(rng.integers(0, 4))

    if direction == 0:
        kernel[kernel_size // 2, :] = 1.0
    elif direction == 1:
        kernel[:, kernel_size // 2] = 1.0
    elif direction == 2:
        np.fill_diagonal(kernel, 1.0)
    else:
        np.fill_diagonal(np.fliplr(kernel), 1.0)

    kernel /= kernel.sum()
    output = cv2.filter2D(image, -1, kernel)
    return output, boxes.copy(), classes.copy()


def cv_gaussian_noise(image, boxes, classes, rng):
    sigma = float(rng.uniform(5.0, 18.0))
    noise = rng.normal(0, sigma, size=image.shape).astype(np.float32)

    output = np.clip(
        image.astype(np.float32) + noise,
        0,
        255,
    ).astype(np.uint8)

    return output, boxes.copy(), classes.copy()


def cv_jpeg_compression(image, boxes, classes, rng):
    quality = int(rng.integers(45, 86))
    success, encoded = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, quality],
    )

    if not success:
        return image.copy(), boxes.copy(), classes.copy()

    output = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    return output, boxes.copy(), classes.copy()

In [ ]:
def cv_affine(image, boxes, classes, rng):
    height, width = image.shape[:2]

    angle = float(rng.uniform(-12, 12))
    scale = float(rng.uniform(0.88, 1.12))
    translate_x = float(rng.uniform(-0.07, 0.07) * width)
    translate_y = float(rng.uniform(-0.07, 0.07) * height)

    matrix = cv2.getRotationMatrix2D(
        (width / 2, height / 2),
        angle,
        scale,
    )
    matrix[0, 2] += translate_x
    matrix[1, 2] += translate_y

    output = cv2.warpAffine(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    new_boxes, keep = transform_boxes(
        boxes,
        matrix,
        width,
        height,
    )

    return output, new_boxes, classes[keep]


def cv_perspective(image, boxes, classes, rng):
    height, width = image.shape[:2]
    jitter = 0.035

    source = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ], dtype=np.float32)

    destination = source.copy()
    destination[:, 0] += rng.uniform(-jitter * width, jitter * width, 4)
    destination[:, 1] += rng.uniform(-jitter * height, jitter * height, 4)

    matrix = cv2.getPerspectiveTransform(
        source,
        destination.astype(np.float32),
    )

    output = cv2.warpPerspective(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    new_boxes, keep = transform_boxes(
        boxes,
        matrix,
        width,
        height,
    )

    return output, new_boxes, classes[keep]


def cv_horizontal_flip(image, boxes, classes, rng):
    height, width = image.shape[:2]
    output = cv2.flip(image, 1)
    new_boxes = boxes.copy()

    if len(new_boxes):
        old_x1 = boxes[:, 0].copy()
        old_x2 = boxes[:, 2].copy()
        new_boxes[:, 0] = width - old_x2
        new_boxes[:, 2] = width - old_x1

    return output, new_boxes, classes.copy()

In [ ]:
def cv_reencode_control(image, boxes, classes, rng):
    """픽셀 변환 없이 OpenCV decode/re-encode 영향만 측정하는 control입니다."""
    return image.copy(), boxes.copy(), classes.copy()


def cv_photometric_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = image.copy(), boxes.copy(), classes.copy()

    if rng.random() < 0.80:
        output, new_boxes, new_classes = cv_brightness_contrast(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.35:
        output, new_boxes, new_classes = cv_gamma(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.30:
        output, new_boxes, new_classes = cv_clahe(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.25:
        output, new_boxes, new_classes = cv_gaussian_noise(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.20:
        output, new_boxes, new_classes = cv_jpeg_compression(
            output, new_boxes, new_classes, rng
        )

    return output, new_boxes, new_classes


def cv_geometric_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = image.copy(), boxes.copy(), classes.copy()

    if rng.random() < 0.80:
        output, new_boxes, new_classes = cv_affine(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.30:
        output, new_boxes, new_classes = cv_perspective(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.45:
        output, new_boxes, new_classes = cv_horizontal_flip(
            output, new_boxes, new_classes, rng
        )

    return output, new_boxes, new_classes


def cv_mixed_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = cv_photometric_combo(
        image, boxes, classes, rng
    )
    output, new_boxes, new_classes = cv_geometric_combo(
        output, new_boxes, new_classes, rng
    )
    return output, new_boxes, new_classes


CV_POLICIES = {
    "reencode_control": cv_reencode_control,
    "brightness_contrast": cv_brightness_contrast,
    "gamma": cv_gamma,
    "clahe": cv_clahe,
    "gaussian_blur": cv_gaussian_blur,
    "motion_blur": cv_motion_blur,
    "gaussian_noise": cv_gaussian_noise,
    "jpeg_compression": cv_jpeg_compression,
    "affine": cv_affine,
    "perspective": cv_perspective,
    "horizontal_flip": cv_horizontal_flip,
    "photometric_combo": cv_photometric_combo,
    "geometric_combo": cv_geometric_combo,
    "mixed_combo": cv_mixed_combo,
}

In [ ]:
def stable_seed(*parts) -> int:
    text = "|".join(map(str, parts))
    return int(hashlib.sha1(text.encode("utf-8")).hexdigest()[:8], 16)


def build_opencv_dataset(
    policy_name: str,
    seed: int,
    apply_probability: float = CV_APPLY_PROBABILITY,
    overwrite: bool = False,
):
    if policy_name not in CV_POLICIES:
        raise KeyError(policy_name)

    policy_fn = CV_POLICIES[policy_name]
    dataset_dir = CV_CACHE_DIR / policy_name / f"seed_{seed}"
    image_dir = dataset_dir / "images" / "train"
    label_dir = dataset_dir / "labels" / "train"
    report_path = dataset_dir / "generation_report.csv"
    yaml_path = dataset_dir / "data.yaml"

    if yaml_path.exists() and report_path.exists() and not overwrite:
        return yaml_path

    if dataset_dir.exists() and overwrite:
        shutil.rmtree(dataset_dir)

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for source_image_path in train_images:
        source_label_path = image_to_label_path(source_image_path)
        image = cv2.imread(str(source_image_path))

        if image is None:
            raise ValueError(f"이미지를 읽지 못했습니다: {source_image_path}")

        height, width = image.shape[:2]
        classes, boxes = read_yolo_label(source_label_path, width, height)

        image_seed = stable_seed(seed, policy_name, source_image_path.name)
        rng = np.random.default_rng(image_seed)
        apply_aug = rng.random() < apply_probability

        if apply_aug:
            augmented_image, augmented_boxes, augmented_classes = policy_fn(
                image, boxes, classes, rng
            )

            # 기하 변환으로 원래 있던 모든 객체가 사라지면 너무 공격적인 샘플이므로 원본으로 fallback합니다.
            if len(classes) > 0 and len(augmented_classes) == 0:
                apply_aug = False
            else:
                output_image_path = image_dir / f"{source_image_path.stem}.jpg"
                success = cv2.imwrite(
                    str(output_image_path),
                    augmented_image,
                    [cv2.IMWRITE_JPEG_QUALITY, CV_OUTPUT_JPEG_QUALITY],
                )

                if not success:
                    raise IOError(f"이미지 저장 실패: {output_image_path}")

                output_label_path = label_dir / f"{source_image_path.stem}.txt"
                output_label_path.write_text(
                    "\n".join(
                        boxes_to_yolo_lines(
                            augmented_classes,
                            augmented_boxes,
                            width,
                            height,
                        )
                    ),
                    encoding="utf-8",
                )

                rows.append({
                    "source_image": source_image_path.name,
                    "output_image": output_image_path.name,
                    "augmented": True,
                    "source_objects": len(classes),
                    "output_objects": len(augmented_classes),
                })

        if not apply_aug:
            output_image_path = image_dir / source_image_path.name
            shutil.copy2(source_image_path, output_image_path)

            output_label_path = label_dir / source_label_path.name
            shutil.copy2(source_label_path, output_label_path)

            rows.append({
                "source_image": source_image_path.name,
                "output_image": output_image_path.name,
                "augmented": False,
                "source_objects": len(classes),
                "output_objects": len(classes),
            })

    generation_df = pd.DataFrame(rows)
    generation_df.to_csv(report_path, index=False, encoding="utf-8-sig")

    # train은 OpenCV static dataset, val은 원본 processed val을 그대로 사용합니다.
    cv_dataset_yaml = {
        "train": str(image_dir.resolve()),
        "val": str(VAL_IMAGE_DIR.resolve()),
        "names": {class_id: name for class_id, name in CLASS_NAMES.items()},
    }

    with open(yaml_path, "w", encoding="utf-8") as file:
        yaml.safe_dump(cv_dataset_yaml, file, allow_unicode=True, sort_keys=False)

    if len(generation_df) != len(train_images):
        raise ValueError("OpenCV dataset의 이미지 수가 원본 train과 다릅니다.")

    return yaml_path

## 9. 실험 카탈로그

증강 조합의 설정을 찾아오기 위한 전체 목록입니다.

In [ ]:
EXPERIMENTS = [
    # Controls
    {
        "id": "B00", "name": "no_aug_auto", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": NO_AUG, "required_args": [],
        "hp_mode": "auto",
        "description": "00번 baseline 재현: 증강 전부 OFF + optimizer auto",
    },
    {
        "id": "B01", "name": "yolo_default_aug_auto", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": None, "required_args": [],
        "hp_mode": "auto",
        "description": "YOLO 기본 augmentation + optimizer auto",
    },
    {
        "id": "B02", "name": "yolo_default_aug_tuned", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": None, "required_args": [],
        "hp_mode": "tuned",
        "description": "YOLO 기본 augmentation + Optuna tuned 하이퍼파라미터",
    },
    {
        "id": "B03", "name": "no_aug_tuned", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": NO_AUG, "required_args": [],
        "hp_mode": "tuned",
        "description": "증강 전부 OFF + Optuna tuned (순수 하이퍼파라미터 효과)",
    },

    # YOLO single-factor screening
    {
        "id": "Y01", "name": "yolo_hsv", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["hsv"], "required_args": ["hsv_h", "hsv_s", "hsv_v"],
        "description": "HSV 색조/채도/밝기 변화",
    },
    {
        "id": "Y02", "name": "yolo_horizontal_flip", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["flip"], "required_args": ["fliplr"],
        "description": "좌우 반전",
    },
    {
        "id": "Y03", "name": "yolo_rotation", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["rotation"], "required_args": ["degrees"],
        "description": "약한 회전",
    },
    {
        "id": "Y04", "name": "yolo_translate", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["translate"], "required_args": ["translate"],
        "description": "객체 위치 이동",
    },
    {
        "id": "Y05", "name": "yolo_scale", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["scale"], "required_args": ["scale"],
        "description": "확대/축소",
    },
    {
        "id": "Y06", "name": "yolo_shear", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["shear"], "required_args": ["shear"],
        "description": "약한 shear 기울임",
    },
    {
        "id": "Y07", "name": "yolo_perspective", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["perspective"], "required_args": ["perspective"],
        "description": "약한 원근 왜곡",
    },
    {
        "id": "Y08", "name": "yolo_mosaic", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic"], "required_args": ["mosaic"],
        "description": "여러 이미지를 한 학습 장면에 구성",
    },
    {
        "id": "Y09", "name": "yolo_mixup", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mixup"], "required_args": ["mixup"],
        "description": "두 이미지와 label을 blending",
    },
    {
        "id": "Y10", "name": "yolo_cutmix", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["cutmix"], "required_args": ["cutmix"],
        "description": "다른 이미지의 직사각형 영역을 붙여 occlusion 생성",
    },

    # YOLO combinations
    {
        "id": "Y11", "name": "yolo_hsv_flip", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["hsv_flip"], "required_args": ["hsv_h", "fliplr"],
        "description": "HSV + 좌우 반전",
    },
    {
        "id": "Y12", "name": "yolo_geometry_combo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["geo_combo"], "required_args": ["degrees", "translate", "scale"],
        "description": "회전/이동/scale/shear/perspective/flip 조합",
    },
    {
        "id": "Y13", "name": "yolo_mosaic_hsv_geo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic_hsv_geo"], "required_args": ["mosaic", "hsv_h", "degrees"],
        "description": "Mosaic + HSV + 약한 geometry",
    },
    {
        "id": "Y14", "name": "yolo_mosaic_mixup_cutmix", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic_mixup_cutmix"], "required_args": ["mosaic", "mixup", "cutmix"],
        "description": "세 가지 multi-image augmentation 조합",
    },
    {
        "id": "Y15", "name": "yolo_balanced_combo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["balanced_combo"], "required_args": ["hsv_h", "degrees", "mosaic"],
        "description": "색/기하/multi-image를 모두 약하게 섞은 조합",
    },

    # OpenCV static augmentation
    *[
        {
            "id": f"C{index:02d}",
            "name": f"opencv_{policy_name}",
            "family": "opencv_single" if policy_name not in {"photometric_combo", "geometric_combo", "mixed_combo"} else "opencv_combo",
            "data_kind": "opencv",
            "cv_policy": policy_name,
            "aug": NO_AUG,
            "required_args": [],
            "description": f"OpenCV static augmentation: {policy_name}",
        }
        for index, policy_name in enumerate(CV_POLICIES.keys(), start=1)
    ],

    # Hybrid: static OpenCV + online YOLO
    {
        "id": "H01", "name": "cv_photo_plus_yolo_geo", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "photometric_combo",
        "aug": YOLO_AUG_CONFIGS["geo_combo"], "required_args": ["degrees", "translate", "scale"],
        "description": "OpenCV photometric + YOLO online geometry",
    },
    {
        "id": "H02", "name": "cv_geo_plus_yolo_hsv", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "geometric_combo",
        "aug": YOLO_AUG_CONFIGS["hsv"], "required_args": ["hsv_h", "hsv_s", "hsv_v"],
        "description": "OpenCV geometry + YOLO online HSV",
    },
    {
        "id": "H03", "name": "cv_mixed_plus_yolo_light", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "mixed_combo",
        "aug": with_no_aug(
            hsv_h=0.010, hsv_s=0.30, hsv_v=0.25,
            degrees=5.0, translate=0.04, scale=0.12,
            fliplr=0.30, mosaic=0.30, close_mosaic=CLOSE_MOSAIC_EPOCHS,
        ),
        "required_args": ["hsv_h", "degrees", "mosaic"],
        "description": "OpenCV mixed + 약한 YOLO online augmentation",
    },
    {
        "id": "H04", "name": "cv_jpeg_plus_yolo_hsv_geo", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "jpeg_compression",
        "aug": with_no_aug(
            hsv_h=0.012, hsv_s=0.40, hsv_v=0.30,
            degrees=7.0, translate=0.05, scale=0.15, fliplr=0.40,
        ),
        "required_args": ["hsv_h", "degrees", "scale"],
        "description": "업로드 압축 품질 저하 + YOLO 색/기하 변화",
    },
]

# hp_mode를 명시하지 않은 실험은 모두 Optuna tuned를 사용합니다.
for experiment in EXPERIMENTS:
    experiment.setdefault("hp_mode", "tuned")

if SMOKE_TEST:
    smoke_ids = {"B00", "B01", "B02", "B03", "Y01", "C01", "H01"}
    EXPERIMENTS = [experiment for experiment in EXPERIMENTS if experiment["id"] in smoke_ids]

planned_experiments_df = pd.DataFrame([
    {
        "id": experiment["id"],
        "name": experiment["name"],
        "family": experiment["family"],
        "hp_mode": experiment["hp_mode"],
        "data_kind": experiment["data_kind"],
        "cv_policy": experiment["cv_policy"],
        "description": experiment["description"],
    }
    for experiment in EXPERIMENTS
])

display(planned_experiments_df)
print("총 본 실험 수:", len(EXPERIMENTS))

# 10. 05번 결과 불러오기 — 하이퍼파라미터 top2 / 증강 top2

05번이 저장한 두 파일을 읽습니다.

```text
05_retune_optuna_augmentation/report/optuna/top2_hyperparameters.json
05_retune_optuna_augmentation/report/summary/top2_augmentations.json
```

둘 다 직접 지정할 수 있는 변수를 맨 위에 두었습니다.
05번을 아직 돌리지 않았다면 여기서 멈추고 안내가 나옵니다.

In [ ]:
# 직접 지정하고 싶을 때 사용하세요.
TOP2_HP_JSON_OVERRIDE = None        # 예: Path(r"C:\path\top2_hyperparameters.json")
TOP2_AUG_IDS_OVERRIDE = None        # 예: ["Y05", "C06"]

YOLO_ROOT = (PROJECT_ROOT / "../models/yolo").resolve()
NB05_REPORT = YOLO_ROOT / "05_retune_optuna_augmentation" / "report"

CONTROL_IDS = ["B00", "B01", "B02", "B03"]


def load_top2_hyperparameters():
    path = (
        Path(TOP2_HP_JSON_OVERRIDE)
        if TOP2_HP_JSON_OVERRIDE
        else NB05_REPORT / "optuna" / "top2_hyperparameters.json"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"05번의 하이퍼파라미터 top2 파일을 찾지 못했습니다:\n  {path}\n"
            "05번을 끝까지 실행했는지 확인하거나 TOP2_HP_JSON_OVERRIDE에 직접 지정하세요."
        )

    payload = json.loads(path.read_text(encoding="utf-8"))
    records = payload.get("hyperparameters", [])

    if not records:
        raise ValueError(f"하이퍼파라미터 기록이 비어 있습니다: {path}")

    return records, payload, path


TOP2_HP, hp_payload, hp_path = load_top2_hyperparameters()

print("하이퍼파라미터 출처:", hp_path)
print("05번이 탐색한 데이터:", hp_payload.get("dataset", "(기록 없음)"))
print()

for record in TOP2_HP:
    print(f"[{record['label']}] trial {record['trial']}  "
          f"multi-seed 평균 {record['multiseed_mAP50_95_mean']:.4f}")

    for key, value in record["params"].items():
        print(f"      {key:>15}: {value}")

    print()

HP_BY_LABEL = {record["label"]: record["params"] for record in TOP2_HP}

if len(TOP2_HP) < 2:
    print("주의: 하이퍼파라미터가 1개뿐이라 교차 실험이 2가지로 줄어듭니다.")


In [ ]:
def load_top2_augmentations():
    if TOP2_AUG_IDS_OVERRIDE:
        return [
            {"rank": index, "label": f"aug{index}", "id": key, "name": key}
            for index, key in enumerate(TOP2_AUG_IDS_OVERRIDE, start=1)
        ], {"note": "TOP2_AUG_IDS_OVERRIDE 직접 지정"}, "직접 지정"

    path = NB05_REPORT / "summary" / "top2_augmentations.json"

    if not path.exists():
        raise FileNotFoundError(
            f"05번의 증강 top2 파일을 찾지 못했습니다:\n  {path}\n"
            "05번을 끝까지 실행했는지 확인하거나 TOP2_AUG_IDS_OVERRIDE에 직접 지정하세요."
        )

    payload = json.loads(path.read_text(encoding="utf-8"))
    records = payload.get("augmentations", [])

    if not records:
        raise ValueError(f"증강 기록이 비어 있습니다: {path}")

    return records, payload, path


TOP2_AUG, aug_payload, aug_path = load_top2_augmentations()

print("증강 출처:", aug_path)

for record in TOP2_AUG:
    score = record.get("mAP50_95")
    score_text = f"  mAP50-95 {score:.4f}" if isinstance(score, (int, float)) else ""
    print(f"  [{record['label']}] {record['id']}  {record.get('name', '')}{score_text}")

gap = aug_payload.get("rank_gap_1_to_2")

if isinstance(gap, (int, float)):
    print()
    print(f"05번에서 1위와 2위의 차이: {gap:.4f}")

    if gap < 0.005:
        print("-> seed 노이즈 수준입니다. 이번 교차 실험에서 순위가 뒤집혀도 이상하지 않습니다.")

if len(TOP2_AUG) < 2:
    print()
    print("주의: 증강이 1개뿐이라 교차 실험이 절반으로 줄어듭니다.")


## 11. 8가지 실험 구성

baseline 4가지 + 교차 4가지입니다.
교차 실험의 ID는 `증강ID_하이퍼파라미터라벨` 형식(`Y05_hp1` 등)이라
결과 표에서 어떤 조합인지 바로 읽을 수 있습니다.

`B02` / `B03`은 대표값인 **hp1**을 사용합니다.

In [ ]:
catalog_by_id = {experiment["id"]: experiment for experiment in EXPERIMENTS}

missing = [record["id"] for record in TOP2_AUG if record["id"] not in catalog_by_id]

if missing:
    raise KeyError(f"카탈로그에 없는 증강 ID: {missing}")

PRIMARY_HP_LABEL = TOP2_HP[0]["label"]

built = []

# ------------------------------------------------------------
# baseline 4가지
# ------------------------------------------------------------
for key, hp_label in [
    ("B00", None),
    ("B01", None),
    ("B02", PRIMARY_HP_LABEL),
    ("B03", PRIMARY_HP_LABEL),
]:
    experiment = dict(catalog_by_id[key])

    if hp_label is None:
        experiment["hp_mode"] = "auto"
        experiment["hp_label"] = "auto"
        experiment["hp_params"] = {}
    else:
        experiment["hp_mode"] = "tuned"
        experiment["hp_label"] = hp_label
        experiment["hp_params"] = HP_BY_LABEL[hp_label]
        experiment["description"] = f"{experiment['description']} ({hp_label})"

    built.append(experiment)

# ------------------------------------------------------------
# 교차 4가지: 증강 top2 x 하이퍼파라미터 top2
# 같은 증강을 연달아 두어 OpenCV static dataset 재생성을 줄입니다.
# ------------------------------------------------------------
for aug_record in TOP2_AUG:
    for hp_record in TOP2_HP:
        experiment = dict(catalog_by_id[aug_record["id"]])

        experiment["id"] = f"{aug_record['id']}_{hp_record['label']}"
        experiment["name"] = f"{experiment['name']}_{hp_record['label']}"
        experiment["family"] = "cross"
        experiment["hp_mode"] = "tuned"
        experiment["hp_label"] = hp_record["label"]
        experiment["hp_params"] = hp_record["params"]
        experiment["description"] = (
            f"{aug_record['label']}({aug_record['id']}) x "
            f"{hp_record['label']}(trial {hp_record.get('trial', '?')})"
        )

        built.append(experiment)

EXPERIMENTS = built

CROSS_IDS = [experiment["id"] for experiment in EXPERIMENTS if experiment["family"] == "cross"]
ORDERED_IDS = CONTROL_IDS + CROSS_IDS

display(
    pd.DataFrame(EXPERIMENTS)[
        ["id", "name", "family", "data_kind", "cv_policy", "hp_label", "description"]
    ]
)

print()
print(f"데이터셋: {DATASET_LABEL}")
print(f"총 {len(EXPERIMENTS)}개 실험 x {EPOCHS} epoch")
print(f"  baseline: {CONTROL_IDS}")
print(f"  교차     : {CROSS_IDS}")


## 12. 실행에 필요한 함수들

`train_one_experiment`만 02번에서 조금 바꿨습니다 — 전역 하나가 아니라 **실험마다 다른 하이퍼파라미터**를 적용합니다.

In [ ]:
def missing_required_args(experiment):
    if not DEFAULT_CFG_DICT:
        return []

    supported = set(DEFAULT_CFG_DICT.keys())
    return [
        argument
        for argument in experiment.get("required_args", [])
        if argument not in supported
    ]


compatibility_rows = []

for experiment in EXPERIMENTS:
    missing = missing_required_args(experiment)
    compatibility_rows.append({
        "id": experiment["id"],
        "name": experiment["name"],
        "supported": len(missing) == 0,
        "missing_required_args": ", ".join(missing),
    })

compatibility_df = pd.DataFrame(compatibility_rows)
display(compatibility_df)

In [ ]:
def dataset_yaml_for_experiment(experiment, seed):
    if experiment["data_kind"] == "processed":
        return RUNTIME_DATA_YAML

    if experiment["data_kind"] == "opencv":
        return build_opencv_dataset(
            experiment["cv_policy"],
            seed=seed,
            apply_probability=CV_APPLY_PROBABILITY,
            overwrite=False,
        )

    raise ValueError(f"알 수 없는 data_kind: {experiment['data_kind']}")

In [ ]:
def extract_metrics(metrics):
    box = metrics.box

    precision = float(getattr(box, "mp", np.nan))
    recall = float(getattr(box, "mr", np.nan))

    if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    speed = getattr(metrics, "speed", {}) or {}

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(box.map50),
        "mAP75": float(box.map75),
        "mAP50_95": float(box.map),
        "inference_ms_per_image": speed.get("inference", np.nan),
    }

In [ ]:
def save_per_class_metrics(metrics, experiment_id, experiment_name, seed):
    maps = np.asarray(metrics.box.maps, dtype=float)

    per_class = pd.DataFrame({
        "class_id": range(len(maps)),
        "class_name": [CLASS_NAMES.get(index, f"class_{index}") for index in range(len(maps))],
        "mAP50_95": maps,
    })

    if len(class_support_df):
        support_cols = [col for col in ["class_name", "train", "val"] if col in class_support_df.columns]
        per_class = per_class.merge(
            class_support_df[support_cols],
            on="class_name",
            how="left",
        )

    path = PER_CLASS_DIR / f"{experiment_id}_{experiment_name}_seed{seed}.csv"
    per_class.to_csv(path, index=False, encoding="utf-8-sig")
    return path

In [ ]:
RESULTS_CSV = SUMMARY_DIR / "experiment_results.csv"


def hp_params_for(experiment):
    """실험별 하이퍼파라미터를 돌려줍니다. 비어 있으면 optimizer='auto'로 학습됩니다."""
    if experiment.get("hp_mode", "tuned") != "tuned":
        return {}

    return dict(experiment.get("hp_params") or {})


def hp_source_for(experiment):
    """결과 표에 남길 하이퍼파라미터 이름표 (auto / hp1 / hp2 ...)."""
    if not hp_params_for(experiment):
        return "auto"

    return experiment.get("hp_label", "optuna_tuned")


def train_one_experiment(experiment, seed=SEED, epochs=None):
    epochs = EPOCHS if epochs is None else epochs

    missing = missing_required_args(experiment)

    if missing:
        return {
            "status": "SKIPPED_UNSUPPORTED",
            "id": experiment["id"],
            "name": experiment["name"],
            "family": experiment["family"],
            "seed": seed,
            "requested_epochs": epochs,
            "hp_mode": experiment.get("hp_mode", "tuned"),
            "hp_source": hp_source_for(experiment),
            "error": f"unsupported args: {missing}",
        }

    dataset_yaml = dataset_yaml_for_experiment(experiment, seed)
    # epoch 수가 다르면 서로 덮어쓰지 않도록 run 이름을 구분합니다.
    run_name = f"{experiment['id']}_{experiment['name']}_seed{seed}_e{epochs}"

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 매우 중요: 모든 실험을 같은 pretrained weight에서 새로 시작합니다.
    model = YOLO(MODEL_NAME)

    train_kwargs = {
        "data": str(dataset_yaml),
        "epochs": epochs,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "patience": PATIENCE,
        "device": DEVICE,
        "workers": WORKERS,
        "seed": seed,
        "deterministic": True,
        # aug=None (B01/B02 처럼 YOLO 기본값을 쓰는 실험)은 Ultralytics 기본
        # close_mosaic=10 이 적용되어, epochs 가 10~15 면 mosaic 이 거의
        # 또는 전부 꺼집니다. 그래서 여기서 기본값을 깔아 둡니다.
        # (aug 딕셔너리에 close_mosaic 이 있으면 아래 update 에서 덮어씁니다)
        "close_mosaic": CLOSE_MOSAIC_EPOCHS,
        # TUNED_PARAMS가 있으면 아래에서 덮어씁니다.
        "optimizer": "auto",
        "amp": True,
        "cache": False,
        "project": str(RUNS_DIR),
        "name": run_name,
        "exist_ok": True,
        "plots": True,
        "verbose": True,
    }

    # 실험마다 다른 하이퍼파라미터를 쓸 수 있습니다 (hp1 / hp2 교차 실험용).
    # B00 / B01은 "auto"이므로 위의 optimizer="auto" 기본값을 그대로 씁니다.
    experiment_hp_source = hp_source_for(experiment)
    experiment_hp_params = hp_params_for(experiment)

    if experiment_hp_params:
        train_kwargs.update(experiment_hp_params)

    filtered_aug, unknown_aug = supported_train_args(experiment["aug"])

    if experiment["aug"] is not None:
        train_kwargs.update(filtered_aug)

    start_time = time.perf_counter()
    train_result = model.train(**train_kwargs)
    train_minutes = (time.perf_counter() - start_time) / 60.0

    save_dir = Path(train_result.save_dir)
    best_pt = save_dir / "weights" / "best.pt"

    if not best_pt.exists():
        raise FileNotFoundError(best_pt)

    history_csv = save_dir / "results.csv"
    actual_epochs = np.nan

    if history_csv.exists():
        history = pd.read_csv(history_csv)
        actual_epochs = len(history)

    best_model = YOLO(str(best_pt))

    val_metrics = best_model.val(
        data=str(dataset_yaml),
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        plots=False,
        verbose=False,
    )

    per_class_path = save_per_class_metrics(
        val_metrics,
        experiment["id"],
        experiment["name"],
        seed,
    )

    row = {
        "status": "OK",
        "id": experiment["id"],
        "name": experiment["name"],
        "family": experiment["family"],
        "description": experiment["description"],
        "seed": seed,
        "data_kind": experiment["data_kind"],
        "cv_policy": experiment["cv_policy"],
        "dataset_yaml": str(dataset_yaml),
        "requested_epochs": epochs,
        "actual_epochs": actual_epochs,
        "train_minutes": train_minutes,
        "best_pt": str(best_pt),
        "save_dir": str(save_dir),
        "per_class_csv": str(per_class_path),
        "unknown_filtered_aug_args": ",".join(unknown_aug),
        "hp_mode": experiment.get("hp_mode", "tuned"),
        "hp_source": experiment_hp_source,
        "hp_params": json.dumps(experiment_hp_params, ensure_ascii=False),
        **extract_metrics(val_metrics),
    }

    # OpenCV static dataset은 용량이 클 수 있으므로 결과 보고서를 보존한 뒤 cache를 정리합니다.
    if experiment["data_kind"] == "opencv" and CLEANUP_CV_DATASET_AFTER_RUN:
        dataset_dir = Path(dataset_yaml).parent
        generation_report = dataset_dir / "generation_report.csv"

        if generation_report.exists():
            persistent_report = SUMMARY_DIR / f"opencv_generation_{experiment['cv_policy']}_seed{seed}.csv"
            shutil.copy2(generation_report, persistent_report)

        shutil.rmtree(dataset_dir, ignore_errors=True)

    del model
    del best_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row

In [ ]:
def load_existing_results():
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame()


def has_successful_run(existing_df, experiment, seed, epochs):
    """같은 (실험, seed, epoch, 하이퍼파라미터 설정) 조합이 이미 성공했는지 확인합니다."""
    if not len(existing_df):
        return False

    required_cols = {"status", "id", "seed"}
    if not required_cols.issubset(existing_df.columns):
        return False

    mask = (
        existing_df["status"].eq("OK")
        & existing_df["id"].eq(experiment["id"])
        & existing_df["seed"].eq(seed)
    )

    # epoch이 다르면 다른 실험입니다. (10 epoch 스크리닝 vs 15 epoch 최종 비교)
    if "requested_epochs" in existing_df.columns:
        mask = mask & existing_df["requested_epochs"].eq(epochs)
    else:
        return False

    # 하이퍼파라미터 설정이 바뀌면(auto <-> optuna_tuned) 같은 표에 둘 수 없으므로
    # 건너뛰지 않고 다시 학습합니다.
    if "hp_source" in existing_df.columns:
        previous_source = existing_df["hp_source"].fillna("auto")
    else:
        previous_source = pd.Series("auto", index=existing_df.index)

    mask = mask & previous_source.eq(hp_source_for(experiment))

    return bool(mask.any())


def run_experiment_list(experiments, seed=SEED, epochs=None):
    epochs = EPOCHS if epochs is None else epochs

    existing_df = load_existing_results()
    new_rows = []

    for index, experiment in enumerate(experiments, start=1):
        print("=" * 100)
        print(
            f"[{index}/{len(experiments)}] {experiment['id']} - {experiment['name']}"
            f" - seed={seed} - {epochs} epoch - {hp_source_for(experiment)}"
        )
        print(experiment["description"])

        if SKIP_COMPLETED and has_successful_run(existing_df, experiment, seed, epochs):
            print("이미 성공한 결과가 있어 건너뜁니다.")
            continue

        try:
            row = train_one_experiment(experiment, seed=seed, epochs=epochs)
        except Exception as error:
            row = {
                "status": "FAILED",
                "id": experiment["id"],
                "name": experiment["name"],
                "family": experiment["family"],
                "seed": seed,
                "requested_epochs": epochs,
                "hp_mode": experiment.get("hp_mode", "tuned"),
                "hp_source": hp_source_for(experiment),
                "error": repr(error),
            }

        new_rows.append(row)

        current_existing = load_existing_results()
        combined = pd.concat(
            [current_existing, pd.DataFrame([row])],
            ignore_index=True,
        )

        # 같은 id+seed가 여러 번 존재하면 가장 최근 행을 유지합니다.
        dedup_keys = [
            key
            for key in ["id", "seed", "requested_epochs", "hp_source"]
            if key in combined.columns
        ]

        if dedup_keys:
            combined = combined.drop_duplicates(
                subset=dedup_keys,
                keep="last",
            )

        combined.to_csv(
            RESULTS_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        display(pd.DataFrame([row]))

    return load_existing_results()

## 13. 학습 실행

8개 x 15 epoch입니다. 중간에 끊겨도 `SKIP_COMPLETED=True`라 이어서 진행됩니다.

`skip` 판정은 (실험 ID, seed, epoch, 하이퍼파라미터 라벨)을 함께 보므로 hp1과 hp2가 섞이지 않습니다.

In [ ]:
if RUN_EXPERIMENTS:
    results_df = run_experiment_list(
        EXPERIMENTS,
        seed=SEED,
        epochs=EPOCHS,
    )

else:
    print("RUN_EXPERIMENTS=False")
    results_df = load_existing_results()

display(results_df)


## 14. 8가지 비교표

In [ ]:
set_korean_font()

results_df = load_existing_results()

compare_df = results_df[
    results_df["status"].eq("OK")
    & results_df["requested_epochs"].eq(EPOCHS)
].drop_duplicates(subset=["id"], keep="last")

BASE_LABELS = {
    "B00": "B00  증강OFF + auto",
    "B01": "B01  기본증강 + auto",
    "B02": f"B02  기본증강 + {PRIMARY_HP_LABEL}",
    "B03": f"B03  증강OFF + {PRIMARY_HP_LABEL}",
}

LABELS = dict(BASE_LABELS)

for aug_record in TOP2_AUG:
    for hp_record in TOP2_HP:
        key = f"{aug_record['id']}_{hp_record['label']}"
        LABELS[key] = f"{key}  {aug_record['label']} x {hp_record['label']}"

order = [key for key in ORDERED_IDS if key in set(compare_df["id"])]
compare_df = compare_df.set_index("id").loc[order].reset_index()
compare_df["설명"] = compare_df["id"].map(LABELS)

display(
    compare_df[
        ["id", "설명", "hp_label", "precision", "recall", "f1", "mAP50", "mAP50_95", "train_minutes"]
    ].round(4)
)

COMPARE_CSV = SUMMARY_DIR / "compare_8_settings.csv"
compare_df.to_csv(COMPARE_CSV, index=False, encoding="utf-8-sig")
print("Saved:", COMPARE_CSV)


## 15. 교차표로 보기

증강(행) x 하이퍼파라미터(열) 표를 만들면
"증강을 바꾼 효과"와 "하이퍼파라미터를 바꾼 효과"를 따로 읽을 수 있습니다.

In [ ]:
cross_rows = []

for aug_record in TOP2_AUG:
    row = {"증강": f"{aug_record['label']} ({aug_record['id']})"}

    for hp_record in TOP2_HP:
        key = f"{aug_record['id']}_{hp_record['label']}"
        match = compare_df[compare_df["id"].eq(key)]
        row[hp_record["label"]] = (
            float(match.iloc[0]["mAP50_95"]) if len(match) else float("nan")
        )

    cross_rows.append(row)

cross_table = pd.DataFrame(cross_rows).set_index("증강")

display(cross_table.round(4))

cross_table.to_csv(SUMMARY_DIR / "cross_table.csv", encoding="utf-8-sig")

if cross_table.notna().all().all() and cross_table.shape == (2, 2):
    hp_labels = list(cross_table.columns)
    aug_labels = list(cross_table.index)

    hp_effect = cross_table[hp_labels[1]] - cross_table[hp_labels[0]]
    aug_effect = cross_table.loc[aug_labels[1]] - cross_table.loc[aug_labels[0]]

    print()
    print(f"하이퍼파라미터를 {hp_labels[0]} -> {hp_labels[1]} 로 바꿨을 때")
    for aug_label, value in hp_effect.items():
        print(f"  {aug_label}: {value:+.4f}")

    print()
    print(f"증강을 {aug_labels[0]} -> {aug_labels[1]} 로 바꿨을 때")
    for hp_label, value in aug_effect.items():
        print(f"  {hp_label}: {value:+.4f}")

    print()
    print("두 효과 모두 ±0.005 이내라면 조합 간 차이가 없다고 보는 편이 안전합니다.")


## 16. 비교 그래프

In [ ]:
set_korean_font()

# ------------------------------------------------------------
# 1) 8가지 주요 지표
# ------------------------------------------------------------
metric_columns = ["mAP50_95", "mAP50", "precision", "recall", "f1"]
metric_labels = ["mAP50-95", "mAP50", "정밀도", "재현율", "F1"]

positions = np.arange(len(compare_df))
bar_width = 0.16

plt.figure(figsize=(15, 6))

for offset, (column, label) in enumerate(zip(metric_columns, metric_labels)):
    plt.bar(
        positions + (offset - 2) * bar_width,
        compare_df[column],
        bar_width,
        label=label,
    )

plt.xticks(positions, compare_df["설명"], rotation=20, ha="right")
plt.ylabel("점수")
plt.title(f"{DATASET_LABEL} · 8가지 설정 비교 ({EPOCHS} epoch)")
plt.legend(ncol=5)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

figure_path = FINAL_DIR / "compare_8_settings.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)


# ------------------------------------------------------------
# 2) B00 대비 변화량
# ------------------------------------------------------------
values = dict(zip(compare_df["id"], compare_df["mAP50_95"]))

if "B00" in values:
    baseline_value = values["B00"]
    gains = compare_df["mAP50_95"] - baseline_value

    plt.figure(figsize=(10, max(5, len(compare_df) * 0.5)))

    bars = plt.barh(
        compare_df["설명"],
        gains,
        color=[
            "#9e9e9e" if abs(g) < 1e-9 else ("#2e7d32" if g > 0 else "#c62828")
            for g in gains
        ],
    )

    for bar, gain in zip(bars, gains):
        plt.text(gain, bar.get_y() + bar.get_height() / 2, f" {gain:+.4f}", va="center")

    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("B00 대비 mAP50-95 변화량")
    plt.title(f"{DATASET_LABEL} · 기준선 대비 변화 · B00={baseline_value:.4f}")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    figure_path = FINAL_DIR / "compare_gain.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)


# ------------------------------------------------------------
# 3) 2x2 효과 분해 (baseline 네 개)
# ------------------------------------------------------------
effects = []

if {"B00", "B03"} <= values.keys():
    effects.append((f"순수 하이퍼파라미터 효과\n(B03 - B00, {PRIMARY_HP_LABEL})",
                    values["B03"] - values["B00"]))

if {"B01", "B02"} <= values.keys():
    effects.append((f"하이퍼파라미터 효과\n(B02 - B01, 기본증강)",
                    values["B02"] - values["B01"]))

if {"B00", "B01"} <= values.keys():
    effects.append(("기본증강 효과\n(B01 - B00, auto)", values["B01"] - values["B00"]))

best_cross = None

if CROSS_IDS:
    available_cross = [key for key in CROSS_IDS if key in values]

    if available_cross:
        best_cross = max(available_cross, key=lambda key: values[key])

        if "B01" in values:
            effects.append((f"최고 교차 조합의 이득\n({best_cross} - B01)",
                            values[best_cross] - values["B01"]))

if effects:
    names = [name for name, _ in effects]
    deltas = [delta for _, delta in effects]

    plt.figure(figsize=(10, 6))

    bars = plt.barh(names, deltas, color=["#2e7d32" if d > 0 else "#c62828" for d in deltas])

    for bar, delta in zip(bars, deltas):
        plt.text(delta, bar.get_y() + bar.get_height() / 2, f" {delta:+.4f}", va="center")

    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("mAP50-95 변화량")
    plt.title(f"{DATASET_LABEL} · 효과 분해")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    figure_path = FINAL_DIR / "effect_decomposition.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)

if best_cross:
    print()
    print(f"교차 조합 중 최고: {best_cross}  mAP50-95 {values[best_cross]:.4f}")


## 17. 클래스별 성능

파손 이미지가 특정 클래스에 몰려 있으면 그 클래스만 크게 떨어집니다.

In [ ]:
set_korean_font()

per_class_frames = []

for _, row in compare_df.iterrows():
    value = row.get("per_class_csv")

    # 빈 값이나 NaN이면 건너뜁니다. Path("")는 현재 폴더가 되어 위험합니다.
    if not isinstance(value, str) or not value.strip():
        continue

    path = Path(value)

    if not path.exists():
        continue

    frame = pd.read_csv(path)
    frame["id"] = row["id"]
    per_class_frames.append(frame)

if per_class_frames:
    per_class_all = pd.concat(per_class_frames, ignore_index=True)

    pivot = per_class_all.pivot_table(
        index="class_name",
        columns="id",
        values="mAP50_95",
    )

    available = [key for key in order if key in pivot.columns]
    pivot = pivot[available].sort_values(available[-1])

    display(pivot.round(4))

    pivot.to_csv(SUMMARY_DIR / "per_class_pivot.csv", encoding="utf-8-sig")

    plt.figure(figsize=(13, max(6, len(pivot) * 0.5)))

    positions = np.arange(len(pivot))
    bar_width = 0.8 / max(1, len(available))

    for offset, key in enumerate(available):
        plt.barh(
            positions + offset * bar_width,
            pivot[key],
            bar_width,
            label=LABELS.get(key, key),
        )

    plt.yticks(positions + bar_width * (len(available) - 1) / 2, pivot.index)
    plt.xlabel("클래스별 mAP50-95")
    plt.title(f"{DATASET_LABEL} · 클래스별 성능")
    plt.legend(fontsize=8, ncol=2)
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    figure_path = FINAL_DIR / "per_class.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)

else:
    print("클래스별 CSV를 찾지 못했습니다.")


## 18. 두 데이터셋을 모두 돌린 뒤

`DATA_IS_ONLY_CLEAN`을 바꿔 두 번 실행하면 결과가 각각 다른 폴더에 남습니다.

```text
06_1_only_clean_experiment/report/summary/compare_8_settings.csv
06_2_clean+20percent_damaged_mixed_experiment/report/summary/compare_8_settings.csv
```

두 CSV를 나란히 놓고 같은 ID끼리 비교하세요.
**장수가 같은 두 데이터셋이므로, 차이는 파손 이미지가 섞인 효과입니다.**

보는 순서를 이렇게 잡으면 발표 자료의 이야기가 자연스럽습니다.

```text
1) B00끼리 비교   파손 이미지가 기본 성능을 얼마나 떨어뜨리는가
2) B01 - B00      증강이 파손 데이터에서 더/덜 도움이 되는가
3) B03 - B00      하이퍼파라미터 튜닝이 파손 데이터에서 더/덜 도움이 되는가
4) 교차 4가지      최적 조합이 두 데이터에서 모두 유지되는가
```

## 마지막으로

차이가 **±0.005~0.02** 수준이면 seed 하나 차이로도 나올 수 있는 범위입니다.
이 실험은 각 설정을 seed 하나로만 돌리므로, 작은 차이로 순위를 단정하지 마세요.
계획대로 추가 Optuna나 증강 탐색은 하지 않으므로,
**결론을 "차이가 없다"로 내리는 것도 정당한 결과**입니다.
